# RAG from Scratch to Production — Episode 18/20
## Auth, rate limiting and access control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/18_auth_rate_limits_access.ipynb)

A RAG app holds people's **private documents** and runs **expensive** AI calls. So three questions matter:
1. **Who are you?** (authentication: passwords and tokens)
2. **How often can you ask?** (rate limiting)
3. **What are you allowed to see?** (access control: documents, and conversations)

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot) answers these with JWT tokens, bcrypt password hashing, an in-memory rate limiter and ownership checks. In this episode we rebuild those pieces **with production's own code**, run them in a small FastAPI server inside the notebook, and try to break them, from the outside, like a user would.

**No API key needed.** We count LLM calls instead of making them.

## 0. Setup

The same library versions as production.

In [ ]:
%pip install -q "fastapi==0.139.0" "uvicorn==0.51.0" "python-jose==3.5.0" "passlib==1.7.4" "bcrypt==4.1.3" requests

In [1]:
import json, time, base64, threading, uuid
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from threading import Lock

import requests, uvicorn
from fastapi import FastAPI, Depends, HTTPException, Request, status
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from jose import jwt, JWTError
from passlib.context import CryptContext
from pydantic import BaseModel

## 1. Production's building blocks

Copied from the production app: password hashing (`app/auth/security.py`), tokens (`app/auth/jwt.py`, `dependencies.py`), and the rate limiter (`app/core/rate_limiter.py`), with production's settings: tokens last 30 minutes, 5 login attempts per minute.

In [2]:
JWT_SECRET_KEY = "change-me-in-production"   # production reads this from the environment
JWT_ALGORITHM = "HS256"
JWT_EXPIRATION_MINUTES = 30

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def hash_password(password: str) -> str:
    return pwd_context.hash(password)

def verify_password(plain: str, hashed: str) -> bool:
    return pwd_context.verify(plain, hashed)

def create_access_token(data: dict, minutes: int = JWT_EXPIRATION_MINUTES) -> str:
    payload = data.copy()
    payload["exp"] = datetime.now(timezone.utc) + timedelta(minutes=minutes)
    return jwt.encode(payload, JWT_SECRET_KEY, algorithm=JWT_ALGORITHM)


class InMemoryRateLimiter:
    """Fixed-window rate limiter keyed by client IP (production's class)."""
    def __init__(self, max_attempts: int, window_seconds: int):
        self.max_attempts, self.window_seconds = max_attempts, window_seconds
        self._hits: dict[str, list[float]] = defaultdict(list)
        self._lock = Lock()

    def check(self, key: str):
        now = time.time()
        with self._lock:
            attempts = [t for t in self._hits[key] if now - t < self.window_seconds]
            if len(attempts) >= self.max_attempts:
                raise HTTPException(status_code=429, detail="Too many attempts. Please try again later.")
            attempts.append(now)
            self._hits[key] = attempts


def rate_limit_dependency(limiter: InMemoryRateLimiter):
    def dependency(request: Request):
        limiter.check(request.client.host if request.client else "unknown")
    return dependency

## 2. The app

Users, documents and conversations live in memory here (production uses PostgreSQL for users and documents, and memory for conversations). Each endpoint follows production's logic:
- `/login`: rate limited, returns a token.
- `/documents/{id}`: only the owner may read it.
- `/chat`: answers using the conversation's history. Production makes 2–3 LLM calls per chat. We count them instead of making them.

In [3]:
USERS = {
    "alice@novatech.example": {"id": 1, "password": hash_password("alice-pass-123")},
    "bob@novatech.example":   {"id": 2, "password": hash_password("bob-pass-456")},
}
DOCUMENTS = {
    10: {"owner": 1, "filename": "alice_salary_review.pdf"},
    20: {"owner": 2, "filename": "bob_notes.txt"},
}
CONVERSATIONS: dict[str, list[dict]] = {}     # production: InMemoryConversationStore, keyed by id only
LLM_CALLS = {"count": 0}

app = FastAPI()
security = HTTPBearer()
login_limiter = InMemoryRateLimiter(max_attempts=5, window_seconds=60)


class LoginRequest(BaseModel):
    email: str
    password: str

class ChatRequest(BaseModel):
    question: str
    conversation_id: str | None = None


def get_current_user(credentials: HTTPAuthorizationCredentials = Depends(security)) -> int:
    try:
        payload = jwt.decode(credentials.credentials, JWT_SECRET_KEY, algorithms=[JWT_ALGORITHM])
        return int(payload["sub"])
    except (JWTError, KeyError, ValueError):
        raise HTTPException(status_code=401, detail="Invalid token")


@app.post("/login", dependencies=[Depends(rate_limit_dependency(login_limiter))])
def login(request: LoginRequest):
    user = USERS.get(request.email)
    if not user or not verify_password(request.password, user["password"]):
        raise HTTPException(status_code=401, detail="Invalid email or password")
    return {"access_token": create_access_token({"sub": str(user["id"])})}


@app.get("/documents/{document_id}")
def get_document(document_id: int, user_id: int = Depends(get_current_user)):
    document = DOCUMENTS.get(document_id)
    if not document:
        raise HTTPException(status_code=404, detail="Document not found")
    if document["owner"] != user_id:                 # production: 403 for someone else's document
        raise HTTPException(status_code=403, detail="You do not have access to this document")
    return document


@app.post("/chat")
def chat(request: ChatRequest, user_id: int = Depends(get_current_user)):
    conversation_id = request.conversation_id or str(uuid.uuid4())
    history = CONVERSATIONS.setdefault(conversation_id, [])   # production: no owner check
    prompt_history = list(history)                             # what the LLM would see
    LLM_CALLS["count"] += 2                                    # answer + hallucination guard
    history.append({"user": user_id, "question": request.question})
    return {"conversation_id": conversation_id, "history_sent_to_llm": prompt_history}


server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8766, log_level="error"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
BASE = "http://127.0.0.1:8766"


def login_as(email, password):
    r = requests.post(f"{BASE}/login", json={"email": email, "password": password})
    return r.status_code, r.json()

def auth(token):
    return {"Authorization": f"Bearer {token}"}

print("Server running")

Server running


## 3. Test 1: tokens

Log in as Alice and look inside the token.

In [4]:
status_code, body = login_as("alice@novatech.example", "alice-pass-123")
ALICE = body["access_token"]
print("Login:", status_code)
print("Token:", ALICE[:60], "...")

# A JWT is three base64 parts: header.payload.signature. Anyone can read the payload:
payload = ALICE.split(".")[1]
print("Payload, decoded without the secret key:", base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)).decode())

Login: 200
Token: eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJzdWIiOiIxIiwiZXhwIjo ...
Payload, decoded without the secret key: {"sub":"1","exp":1795536759}


A JWT is **signed, not encrypted**. Anyone holding a token can read what's inside, so never put secrets in it. Production stores only the user id (`sub`) and expiry (`exp`). ✅

What the signature protects against: **changing** the contents. Let's try to become Bob by editing the user id, and try an expired token.

In [5]:
header, payload, signature = ALICE.split(".")
claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
claims["sub"] = "2"                                     # pretend to be Bob
forged_payload = base64.urlsafe_b64encode(json.dumps(claims).encode()).decode().rstrip("=")
forged = f"{header}.{forged_payload}.{signature}"

expired = create_access_token({"sub": "1"}, minutes=-1)

for label, token in [("Alice's real token", ALICE), ("forged: user id changed to Bob", forged), ("expired token", expired)]:
    r = requests.get(f"{BASE}/documents/10", headers=auth(token))
    print(f"{label:<32} → {r.status_code}")

Alice's real token               → 200
forged: user id changed to Bob   → 401
expired token                    → 401


Changing a single character of the payload breaks the signature: **401**. Expired tokens are rejected too. The token design is sound, **as long as `JWT_SECRET_KEY` stays secret and long**: anyone who has it can sign any token they like.

## 4. Test 2: whose documents can you read?

In [6]:
_, body = login_as("bob@novatech.example", "bob-pass-456")
BOB = body["access_token"]

for label, token, doc_id in [("Alice reads her own document", ALICE, 10),
                             ("Bob reads Alice's document", BOB, 10),
                             ("Bob reads a document that doesn't exist", BOB, 99)]:
    r = requests.get(f"{BASE}/documents/{doc_id}", headers=auth(token))
    print(f"{label:<42} → {r.status_code} {r.json()}")

Alice reads her own document               → 200 {'owner': 1, 'filename': 'alice_salary_review.pdf'}
Bob reads Alice's document                 → 403 {'detail': 'You do not have access to this document'}
Bob reads a document that doesn't exist    → 404 {'detail': 'Document not found'}


The ownership check works: Bob can't read Alice's document. ✅

One small leak: Bob gets **403** ("you can't see this") for Alice's document, but **404** ("doesn't exist") for a made-up id. By trying ids one by one, he can learn **which document ids exist**, and roughly how many documents other users have. The usual fix is to return 404 for both: "not found, or not yours".

And in Episode 6, retrieval was filtered by the user's own documents. That's the check that matters most for RAG: a question can only ever be answered from the asker's documents.

## 5. Test 3: the login rate limit

Someone tries to guess Alice's password:

In [7]:
for attempt in range(1, 8):
    status_code, body = login_as("alice@novatech.example", f"guess-{attempt}")
    print(f"attempt {attempt}: {status_code} {body.get('detail', '')}")

attempt 1: 401 Invalid email or password


attempt 2: 401 Invalid email or password


attempt 3: 401 Invalid email or password
attempt 4: 429 Too many attempts. Please try again later.
attempt 5: 429 Too many attempts. Please try again later.
attempt 6: 429 Too many attempts. Please try again later.
attempt 7: 429 Too many attempts. Please try again later.


Password guessing is slowed down: **429 Too Many Attempts**. ✅

But look at **when** it kicked in: on the **4th** guess, not the 6th. Alice's and Bob's **successful** logins a moment ago came from the same IP in the same minute, and they count too: 2 logins + 3 guesses = 5.

But the limit is counted per **IP address**, and it counts **every** attempt. Bob is on the same office network as the attacker (in this notebook, everyone is `127.0.0.1`), and now types his **correct** password:

In [8]:
status_code, body = login_as("bob@novatech.example", "bob-pass-456")
print(f"Bob, correct password: {status_code} {body.get('detail', '')}")

Bob, correct password: 429 Too many attempts. Please try again later.


**Bob is locked out by someone else's attempts.** Everyone behind one shared IP (an office network, a VPN, a mobile carrier) shares one budget of 5 logins a minute, successful ones included.

Common fixes: count only **failed** attempts, and key them by **IP + account** (to stop guessing one password) plus a looser limit per IP (to stop guessing many). Also check what IP the app actually sees behind your hosting provider's proxy: if every request appears to come from the proxy, everyone shares one budget.

Waiting a minute for the window to clear:

In [9]:
time.sleep(61)
_, body = login_as("bob@novatech.example", "bob-pass-456")
BOB = body["access_token"]
print("Bob can log in again:", "access_token" in body)

Bob can log in again: True


## 6. Test 4: how often can you ask?

Production only rate-limits `/login` and `/register`. Each `/chat` costs 2–3 LLM calls. What does one user sending 100 questions cost?

In [10]:
LLM_CALLS["count"] = 0
start = time.perf_counter()
codes = [requests.post(f"{BASE}/chat", json={"question": f"question {i}"}, headers=auth(BOB)).status_code
         for i in range(100)]
print(f"100 chats in {time.perf_counter() - start:.1f}s → status codes: {set(codes)}")
print(f"LLM calls triggered: {LLM_CALLS['count']}")

100 chats in 0.1s → status codes: {200}
LLM calls triggered: 200


Nothing stops it: 100 requests, **200 LLM calls**. With a paid API that's a cost problem. With a **free tier** it's an availability problem: production's bug #26 was the Gemini free-tier quota running out. One user (or one buggy script) can spend the quota for **everyone**.

The fix is a second limiter on `/chat` (and `/upload`), keyed by **user id** rather than IP, so it follows the account:

In [11]:
chat_limiter = InMemoryRateLimiter(max_attempts=20, window_seconds=60)

@app.post("/chat-limited")
def chat_limited(request: ChatRequest, user_id: int = Depends(get_current_user)):
    chat_limiter.check(f"user:{user_id}")          # keyed by the account, not the network
    return chat(request, user_id)

codes = [requests.post(f"{BASE}/chat-limited", json={"question": f"q{i}"}, headers=auth(BOB)).status_code
         for i in range(25)]
print(f"25 chats → {codes.count(200)} answered, {codes.count(429)} rejected with 429")

25 chats → 20 answered, 5 rejected with 429


## 7. Test 5: whose conversation is it?

Alice has a private conversation:

In [12]:
r = requests.post(f"{BASE}/chat", headers=auth(ALICE),
                  json={"question": "I'm on a performance improvement plan. When is my salary review?"})
alice_conversation = r.json()["conversation_id"]
print("Alice's conversation id:", alice_conversation)

Alice's conversation id: c56b0100-ed38-4c5c-b734-6abeee0727b8


Production's conversation store is keyed by the conversation id **alone**. It never records who owns a conversation. What if Bob sends a request with Alice's conversation id?

In [13]:
r = requests.post(f"{BASE}/chat", headers=auth(BOB),
                  json={"question": "What did I ask about before?", "conversation_id": alice_conversation})
print("Bob's request is answered with this history in the prompt:")
print(json.dumps(r.json()["history_sent_to_llm"], indent=2))

Bob's request is answered with this history in the prompt:
[
  {
    "user": 1,
    "question": "I'm on a performance improvement plan. When is my salary review?"
  }
]


**Alice's private question goes into the prompt for Bob's request.** Production's query rewriter and prompt both include the conversation history (Episodes 9 and 13), so the LLM could repeat it back to him, and Bob's messages are also added to Alice's conversation.

How likely is this? The ids are random UUIDs, so Bob can't **guess** one. He'd need it to **leak**: in a log line, a shared screenshot, a URL, a support ticket. That's why this is called **defense in depth**: the random id is one layer, but the server should also check ownership, the same way it already does for documents.

The fix:

In [14]:
CONVERSATION_OWNERS: dict[str, int] = {}

@app.post("/chat-owned")
def chat_owned(request: ChatRequest, user_id: int = Depends(get_current_user)):
    if request.conversation_id:
        if CONVERSATION_OWNERS.get(request.conversation_id) != user_id:
            raise HTTPException(status_code=404, detail="Conversation not found")
    response = chat(request, user_id)
    CONVERSATION_OWNERS.setdefault(response["conversation_id"], user_id)
    return response

r = requests.post(f"{BASE}/chat-owned", headers=auth(ALICE), json={"question": "When is my salary review?"})
alice_conversation = r.json()["conversation_id"]
r = requests.post(f"{BASE}/chat-owned", headers=auth(BOB),
                  json={"question": "What did I ask about before?", "conversation_id": alice_conversation})
print("Bob with Alice's conversation id →", r.status_code, r.json())

Bob with Alice's conversation id → 404 {'detail': 'Conversation not found'}


In [15]:
server.should_exit = True   # stop the notebook's server

## 8. What production does

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot):
- ✅ Hashes passwords with **bcrypt**, issues **signed JWTs** that expire after 30 minutes, and keeps only the user id in them.
- ✅ Checks **document ownership**, and filters retrieval by the user's own documents (Episode 6). It returns 403 rather than 404 for other users' documents, which reveals which ids exist.
- ⚠️ Rate-limits only `/login` and `/register`, per IP, counting successful logins too. `/chat` and `/upload` have **no limit**, so one user can spend the shared LLM quota.
- ⚠️ Stores conversations by id **without an owner**, so a leaked conversation id exposes that conversation's history. The store also never removes old conversations.
- ℹ️ The rate limiter and the conversation store live in **memory**: they reset on restart and aren't shared between processes (the rate limiter's own comment says so).

## Recap

| Question | Production | What we found | Fix |
|---|---|---|---|
| Who are you? | bcrypt + signed JWT | Forged and expired tokens rejected ✅ | Keep the secret key long and secret |
| Whose documents? | Ownership check | Works ✅, but 403 vs 404 reveals which ids exist | Return 404 for both |
| How many logins? | 5/min per IP | Stops guessing ✅, but locks out coworkers on the same network | Count failures only, key by IP + account |
| How many chats? | No limit | 100 chats → 200 LLM calls | Per-user limit on `/chat` and `/upload` |
| Whose conversation? | No owner stored | A leaked id exposes someone's history | Store the owner, check it on every request |

**The rule:** check ownership on **every** object a user can name in a request (documents, conversations, uploads), and limit **every** endpoint that costs money, not just login.

**Next, Episode 19: PostgreSQL, Alembic and Docker**, making the app's data and setup survive restarts and redeploys.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)